# GLM-4.1V-9B-Thinking QLoRA Finetune — DDP 2xT4, HF dataset, HF-only store

**Model:** `Rainnighttram/GLM-4.1V-9B-Thinking-bnb-4bit` (9B pre-quantized 4-bit) + LoRA r16. Each torchrun rank loads the full 4-bit model onto its own T4 (`device_map={'': rank}`) — true data-parallel, both GPUs train.

**Data:** HF dataset `ShivRamSaud/astroclimb_train` (`train.csv`, OLD 10k, base64 obj_1/obj_2) → `val 1800 balanced` + `train 8200` (seed 42, same split as Gemma/SmolVLM runs).

**Prompt:** same SYSTEM_PROMPT as all other runs (comparability); SFT answer-only, no thinking trigger, no few-shot. Single user turn per sample.

**Store:** local `/kaggle/working/glm41v_out` is scratch only. Durable state = NEW HF repo (auto-created): `checkpoints/checkpoint-N` every save + `adapter_best` + run files. Rerun continues automatically from the latest HF checkpoint.

**Stop:** early stopping (patience 2 on eval_loss) + `load_best_model_at_end`; best adapter pushed at end. Pace ~30-45s/opt-step → ~500-700 steps + 2-3 evals per 9h session.


In [1]:
# --- 0. Config (single source; exported to train script via env) ---
import os
MODEL_ID = 'Rainnighttram/GLM-4.1V-9B-Thinking-bnb-4bit'
HF_REPO_NAME = 'glm41v-9b-lora-astroclimb'
HF_TRAIN_DATASET = 'ShivRamSaud/astroclimb_train'
EPOCHS = 3
MAX_STEPS = 1536
BATCH_SIZE = 1
GRAD_ACCUM = 8
LR = 1e-4
LORA_R = 16
LORA_ALPHA = 32
MAX_SEQ_LEN = 2048
CAP_CHARS = 1500
SAVE_STEPS = 100
EVAL_STEPS = 200
PATIENCE = 2
OUT_DIR = '/kaggle/working/glm41v_out'
FINAL_DIR = '/kaggle/working/glm41v_final'
EVAL_DIR = '/kaggle/working/glm41v_eval'
os.environ['MODEL_ID'] = MODEL_ID
os.environ['HF_REPO_NAME'] = HF_REPO_NAME
os.environ['HF_TRAIN_DATASET'] = HF_TRAIN_DATASET
os.environ['EPOCHS'] = str(EPOCHS)
os.environ['MAX_STEPS'] = str(MAX_STEPS)
os.environ['BATCH_SIZE'] = str(BATCH_SIZE)
os.environ['GRAD_ACCUM'] = str(GRAD_ACCUM)
os.environ['LR'] = str(LR)
os.environ['LORA_R'] = str(LORA_R)
os.environ['LORA_ALPHA'] = str(LORA_ALPHA)
os.environ['MAX_SEQ_LEN'] = str(MAX_SEQ_LEN)
os.environ['CAP_CHARS'] = str(CAP_CHARS)
os.environ['SAVE_STEPS'] = str(SAVE_STEPS)
os.environ['EVAL_STEPS'] = str(EVAL_STEPS)
os.environ['PATIENCE'] = str(PATIENCE)
os.environ['OUT_DIR'] = OUT_DIR
os.environ['FINAL_DIR'] = FINAL_DIR
os.environ['EVAL_DIR'] = EVAL_DIR
print('eff batch = 1 x 8 accum x 2 ranks = 16; steps/ep = 8200/16 = 512')
print('plan: setup ~40min, ~500-700 opt steps + evals every 200 per 9h session')


eff batch = 1 x 8 accum x 2 ranks = 16; steps/ep = 8200/16 = 512
plan: setup ~40min, ~500-700 opt steps + evals every 200 per 9h session


In [2]:
# --- 1. Setup (Kaggle T4 x2: torch>=2.5 + transformers>=4.52, auto-upgrade) ---
import os, json, time, re, gc, base64, sys, subprocess
from pathlib import Path
from io import BytesIO
import pandas as pd, numpy as np
from PIL import Image
from tqdm import tqdm
import importlib.metadata as _im
from packaging import version as _pv
import importlib as _il
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
try:
    _drv = subprocess.check_output(['nvidia-smi', '--query-gpu=driver_version', '--format=csv,noheader'], text=True).strip().splitlines()[0]
    _drv_major = int(_drv.split('.')[0])
except Exception:
    _drv_major = 0
print('nvidia driver major: ' + str(_drv_major))
try:
    _torch_v = _im.version('torch')
except Exception: _torch_v = '0.0.0'
print('installed torch ' + _torch_v)
if _pv.parse(_torch_v) < _pv.parse('2.5.0'):
    _cu = 'cu121' if _drv_major >= 525 else 'cu118'
    print('Upgrading torch to 2.5.1+' + _cu + ' ...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'torch==2.5.1', 'torchaudio==2.5.1', 'torchvision==0.20.1', '--index-url', 'https://download.pytorch.org/whl/' + _cu])
    for m in list(sys.modules.keys()):
        if m.split('.')[0] in ('torch', 'torchvision', 'torchaudio', 'transformers', 'peft', 'accelerate', 'typing_extensions', 'bitsandbytes'): del sys.modules[m]
    _il.invalidate_caches()
import torch
print('torch ' + torch.__version__ + ' cuda ' + str(torch.cuda.is_available()))
assert torch.cuda.is_available(), 'Enable GPU: Settings -> Accelerator -> GPU T4 x2'
assert torch.cuda.device_count() >= 2, 'Need 2xT4'
print([torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
_need = _pv.parse(_im.version('transformers')) < _pv.parse('4.52.0')
print('transformers ' + _im.version('transformers') + ' need>=4.52: ' + str(_need))
if _need:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'transformers>=4.52', 'accelerate', 'peft', 'bitsandbytes>=0.46.1'])
    for m in list(sys.modules.keys()):
        if m.split('.')[0] in ('transformers', 'peft', 'accelerate', 'typing_extensions', 'bitsandbytes'): del sys.modules[m]
    _il.invalidate_caches()
import transformers; print('transformers ' + transformers.__version__)
import peft; print('peft ' + peft.__version__)
try:
    import bitsandbytes as _bnb; print('bitsandbytes ' + _bnb.__version__)
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'bitsandbytes>=0.46.1'])
    import bitsandbytes as _bnb; print('bitsandbytes installed ' + _bnb.__version__)
try:
    import torchao as _ta
    print('stale torchao ' + _ta.__version__ + ' present - removing (breaks peft, unused here)...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'])
    for m in list(sys.modules.keys()):
        if m.split('.')[0] == 'torchao': del sys.modules[m]
    print('torchao removed')
except ImportError:
    print('no torchao present (fine)')
def _datasets_usable():
    try:
        import datasets as _d
        return _pv.parse(_d.__version__) >= _pv.parse('3.0.0')
    except Exception:
        return False
if not _datasets_usable():
    print('Repairing datasets/pyarrow install...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U', '--force-reinstall', '--no-deps', 'datasets', 'pyarrow'])
    for m in list(sys.modules.keys()):
        if m.split('.')[0] in ('datasets', 'pyarrow', 'dill', 'multiprocess', 'fsspec', 'huggingface_hub'): del sys.modules[m]
    _il.invalidate_caches()
if not _datasets_usable():
    print('datasets still broken - removing (Trainer works without it)...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'uninstall', '-y', 'datasets'])
    for m in list(sys.modules.keys()):
        if m.split('.')[0] == 'datasets': del sys.modules[m]
    print('datasets removed')
else:
    import datasets; print('datasets ' + datasets.__version__)


nvidia driver major: 580
installed torch 2.0.0
Upgrading torch to 2.5.1+cu121 ...


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf 23.8.0 requires cupy-cuda11x>=12.0.0, which is not installed.
cuml 23.8.0 requires cupy-cuda11x>=12.0.0, which is not installed.
dask-cudf 23.8.0 requires cupy-cuda11x>=12.0.0, which is not installed.
apache-beam 2.46.0 requires dill<0.3.2,>=0.3.1.1, but you have dill 0.3.7 which is incompatible.
apache-beam 2.46.0 requires pyarrow<10.0.0,>=3.0.0, but you have pyarrow 11.0.0 which is incompatible.
cudf 23.8.0 requires pandas<1.6.0dev0,>=1.3, but you have pandas 2.0.3 which is incompatible.
cudf 23.8.0 requires protobuf<5,>=4.21, but you have protobuf 3.20.3 which is incompatible.
cuml 23.8.0 requires dask==2023.7.1, but you have dask 2023.12.0 which is incompatible.
cuml 23.8.0 requires distributed==2023.7.1, but you have distributed 2023.12.0 which is incompatible.
dask-cudf 23.8.0 requires dask==2023.7.1, b

torch 2.5.1+cu121 cuda True
['Tesla T4', 'Tesla T4']
transformers 4.36.0 need>=4.52: True


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cudf 23.8.0 requires cupy-cuda11x>=12.0.0, which is not installed.
apache-beam 2.46.0 requires dill<0.3.2,>=0.3.1.1, but you have dill 0.3.7 which is incompatible.
apache-beam 2.46.0 requires pyarrow<10.0.0,>=3.0.0, but you have pyarrow 11.0.0 which is incompatible.
dask-cuda 23.8.0 requires dask==2023.7.1, but you have dask 2023.12.0 which is incompatible.
dask-cuda 23.8.0 requires distributed==2023.7.1, but you have distributed 2023.12.0 which is incompatible.
dask-cuda 23.8.0 requires pandas<1.6.0dev0,>=1.3, but you have pandas 2.0.3 which is incompatible.
dask-cudf 23.8.0 requires dask==2023.7.1, but you have dask 2023.12.0 which is incompatible.
dask-cudf 23.8.0 requires distributed==2023.7.1, but you have distributed 2023.12.0 which is incompatible.
dask-cudf 23.8.0 requires pandas<1.6.0dev0,>=1.3, but 

transformers 5.17.0


/opt/conda/lib/python3.10/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.24.3
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


peft 0.20.0
bitsandbytes 0.50.2
no torchao present (fine)
Repairing datasets/pyarrow install...
datasets still broken - removing (Trainer works without it)...
Found existing installation: datasets 5.0.1
Uninstalling datasets-5.0.1:
  Successfully uninstalled datasets-5.0.1
datasets removed


In [3]:
# --- 1b. HF login + auto-create NEW repo (results live here, not Kaggle) ---
from huggingface_hub import login
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
        print('HF_TOKEN from Kaggle Secrets')
    except Exception as e: print('No HF_TOKEN: ' + str(e))
assert HF_TOKEN, 'Add HF_TOKEN (write) to Kaggle Secrets + Internet ON'
login(token=HF_TOKEN)
os.environ['HF_TOKEN'] = HF_TOKEN
from huggingface_hub import HfApi
try:
    _who = HfApi().whoami()['name']
    HF_REPO_ID = _who + '/' + HF_REPO_NAME
except Exception:
    HF_REPO_ID = HF_REPO_NAME
HfApi().create_repo(repo_id=HF_REPO_ID, private=True, exist_ok=True)
os.environ['HF_REPO_ID'] = HF_REPO_ID
print('Repo ready -> ' + HF_REPO_ID + ' (checkpoints/ + adapter_best + run files land here)')


HF_TOKEN from Kaggle Secrets
Repo ready -> ShivRamSaud/glm41v-9b-lora-astroclimb (checkpoints/ + adapter_best + run files land here)


In [4]:
%%writefile /kaggle/working/train_glm41v.py
import os, json, time, re, gc, base64, random
from pathlib import Path
from io import BytesIO
import pandas as pd, numpy as np
from PIL import Image
from tqdm import tqdm
import torch
def _env(k, d):
    return os.environ.get(k, d)
RANK = int(os.environ.get('RANK', '0'))
WORLD = int(os.environ.get('WORLD_SIZE', '1'))
LOCAL_RANK = int(os.environ.get('LOCAL_RANK', '0'))
IS_MAIN = (RANK == 0)
def _log(m):
    if IS_MAIN:
        print(m, flush=True)
MODEL_ID = _env('MODEL_ID', 'Rainnighttram/GLM-4.1V-9B-Thinking-bnb-4bit')
HF_REPO_ID = os.environ['HF_REPO_ID']
HF_TOKEN = os.environ.get('HF_TOKEN')
HF_TRAIN_DATASET = _env('HF_TRAIN_DATASET', 'ShivRamSaud/astroclimb_train')
OUT_DIR = _env('OUT_DIR', '/kaggle/working/glm41v_out')
FINAL_DIR = _env('FINAL_DIR', '/kaggle/working/glm41v_final')
EVAL_DIR = _env('EVAL_DIR', '/kaggle/working/glm41v_eval')
EPOCHS = int(_env('EPOCHS', '3'))
MAX_STEPS = int(_env('MAX_STEPS', '1536'))
BATCH = int(_env('BATCH_SIZE', '1'))
ACCUM = int(_env('GRAD_ACCUM', '8'))
LR = float(_env('LR', '1e-4'))
LORA_R = int(_env('LORA_R', '16'))
LORA_A = int(_env('LORA_ALPHA', '32'))
MAX_SEQ = int(_env('MAX_SEQ_LEN', '2048'))
CAP = int(_env('CAP_CHARS', '1500'))
SAVE_STEPS = int(_env('SAVE_STEPS', '100'))
EVAL_STEPS = int(_env('EVAL_STEPS', '200'))
PATIENCE = int(_env('PATIENCE', '2'))
LABEL_COLS = ['same_figure', 'same_paper', 'related_papers', 'unrelated_papers']
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
torch.cuda.set_device(LOCAL_RANK)
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(EVAL_DIR, exist_ok=True)
_log('rank ' + str(RANK) + '/' + str(WORLD) + ' local_gpu ' + str(LOCAL_RANK) + ' model ' + MODEL_ID)
_log('script rev r11 model-first')
from huggingface_hub import login, HfApi, hf_hub_download, snapshot_download
if HF_TOKEN:
    login(token=HF_TOKEN)
if IS_MAIN:
    HfApi().create_repo(repo_id=HF_REPO_ID, private=True, exist_ok=True)
    _log('repo ready ' + HF_REPO_ID)
from transformers import AutoProcessor
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
_log('processor ' + type(processor).__name__)
try:
    from transformers import Glm4vForConditionalGeneration as ModelClass
    _log('Using Glm4vForConditionalGeneration')
except ImportError:
    from transformers import AutoModelForImageTextToText as ModelClass
    _log('Using AutoModelForImageTextToText fallback')
base = ModelClass.from_pretrained(MODEL_ID, device_map={'': LOCAL_RANK}, torch_dtype=torch.bfloat16, trust_remote_code=True)
_log('base on cuda:' + str(LOCAL_RANK) + ' ' + type(base).__name__)
from peft import LoraConfig, get_peft_model
try:
    from peft import prepare_model_for_kbit_training
    base = prepare_model_for_kbit_training(base)
    _log('kbit prepared')
except Exception as _e:
    base.config.use_cache = False
    _log('kbit prep skipped: ' + str(_e)[:160])
import torch.nn as _nn
_WANT = ('q_proj', 'k_proj', 'v_proj', 'o_proj', 'qkv_proj', 'query_key_value', 'gate_proj', 'up_proj', 'down_proj', 'fc1', 'fc2', 'dense')
_found = sorted({_n.split('.')[-1] for _n, _m in base.named_modules() if isinstance(_m, _nn.Linear) and _n.split('.')[-1] in _WANT})
_targets = _found or ['q_proj', 'v_proj']
_log('lora targets: ' + str(_targets))
lora_config = LoraConfig(r=LORA_R, lora_alpha=LORA_A, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM', target_modules=_targets)
model = get_peft_model(base, lora_config)
if IS_MAIN:
    model.print_trainable_parameters()
model.config.use_cache = False
_csv = '/kaggle/input/competitions/astroclimb/train.csv'
_PARQ = '/kaggle/working/_shared/train_10k.parquet'
_READY = '/kaggle/working/_shared/READY'
_got_local = Path(_csv).exists()
def _read_parquet_chunked(_p):
    import pyarrow.parquet as _pq
    _parts = []
    for _b in _pq.ParquetFile(_p).iter_batches(batch_size=2000):
        _parts.append(_b.to_pandas())
    _df = pd.concat(_parts)
    del _parts
    gc.collect()
    return _df
if RANK == 0:
    os.makedirs('/kaggle/working/_shared', exist_ok=True)
    if not Path(_READY).exists():
        if not _got_local:
            _csv = hf_hub_download(repo_id=HF_TRAIN_DATASET, filename='train.csv', repo_type='dataset')
            _log('downloaded train.csv from ' + HF_TRAIN_DATASET)
        else:
            _log('using attached competition file')
        try:
            _chunks = []
            for _ch in pd.read_csv(_csv, chunksize=2000):
                _chunks.append(_ch)
            _full = pd.concat(_chunks)
            del _chunks
        except Exception as _e:
            print('chunked read failed, plain read: ' + str(_e)[:160], flush=True)
            _full = pd.read_csv(_csv)
        gc.collect()
        _full.to_parquet(_PARQ)
        Path(_READY).touch()
        train = _full
        del _full
    else:
        train = _read_parquet_chunked(_PARQ)
else:
    _waited = 0
    while not Path(_READY).exists():
        time.sleep(10)
        _waited += 10
        if _waited > 1800:
            raise RuntimeError('timed out waiting for rank0 data prep')
    train = _read_parquet_chunked(_PARQ)
try:
    import pyarrow as _pa
    for _c in ['obj_1', 'obj_2']:
        train[_c] = train[_c].astype(pd.ArrowDtype(_pa.large_string()))
    _log('obj cols -> large_string[pyarrow]')
except Exception as _e:
    _log('large_string cast skipped: ' + str(_e)[:160])
_log('rows ' + str(len(train)) + ' cols ' + str(len(train.columns)))
train['label'] = train[LABEL_COLS].idxmax(axis=1)
def is_image_str(s):
    if hasattr(s, 'as_py'):
        s = s.as_py()
    if s is None:
        return False
    try:
        if pd.isna(s):
            return False
    except Exception:
        return False
    if not isinstance(s, str):
        return False
    if len(s) < 200:
        return False
    return s.strip().startswith('iVBORw0KGgo')
def convert_str_to_PIL(img_str):
    if hasattr(img_str, 'as_py'):
        img_str = img_str.as_py()
    return Image.open(BytesIO(base64.b64decode(img_str))).convert('RGB')
train['obj_1_is_img'] = train['obj_1'].apply(is_image_str)
train['obj_2_is_img'] = train['obj_2'].apply(is_image_str)
train['pair_type'] = train.apply(lambda r: ('IMG' if r['obj_1_is_img'] else 'TXT') + '-' + ('IMG' if r['obj_2_is_img'] else 'TXT'), axis=1)
def sample_balanced(df, n_per_class=450, seed=42):
    parts = []
    for _lab in LABEL_COLS:
        _sub = df[df['label'] == _lab]
        if _lab == 'same_figure':
            parts.append(_sub[_sub['pair_type'] == 'TXT-IMG'].sample(n=n_per_class, random_state=seed))
        else:
            _pp = n_per_class // 3
            for _pt in ['TXT-IMG', 'IMG-IMG', 'TXT-TXT']:
                parts.append(_sub[_sub['pair_type'] == _pt].sample(n=_pp, random_state=seed))
    return pd.concat(parts).sample(frac=1, random_state=seed)
val = sample_balanced(train, 450, 42)
_vmask = train.index.isin(val.index)
train_pos = np.flatnonzero(~_vmask)
val_pos = np.flatnonzero(_vmask)
assert len(val_pos) == 1800 and len(train_pos) == 8200, 'split size wrong'
del val, _vmask
gc.collect()
_log('val n=' + str(len(val_pos)) + ' train n=' + str(len(train_pos)) + ' (shared frame, no copies)')
SYSTEM_PROMPT = '''You are an expert in astrophysics figures and captions. Given Object A and Object B (each is either a figure image or a caption text), classify their relation into exactly ONE label based ONLY on what you see/read - no DOI or metadata is provided.\n
\n
Classes:\n
- same_figure: The caption directly describes the figure in front of you. Visual elements (axes, labels, numbers, morphology) are mentioned verbatim in the text, or the text reads like \"Figure X shows...\" matching the image.\n
- same_paper: Same study, different figures. Similar writing style, same instruments/datasets/authors hinted in text, or visual style (fonts, colors, layout) is consistent, but NOT a direct caption-figure match.\n
- related_papers: Different papers where one builds on the other. Overlapping methods, shared datasets, or a figure/caption that looks like a cited prior result, but style/authors differ.\n
- unrelated_papers: No clear link. Different topics, instruments, scales, or writing/visual style with no overlap.\n
\n
Base your decision only on visual and textual content. Do not assume same_figure is impossible for any pair type - judge from alignment.\n
Output ONLY the lowercase label (e.g., related_papers), no explanation, no punctuation.\n
'''
def build_user_content(row):
    parts = []
    for _col, _name in [('obj_1', 'Object A'), ('obj_2', 'Object B')]:
        _s = row[_col]
        if row[_col + '_is_img']:
            parts.append({'name': _name, 'is_img': True, 'pil': convert_str_to_PIL(_s)})
        else:
            parts.append({'name': _name, 'is_img': False, 'text': str(_s)[:CAP]})
    return parts
def build_messages(row):
    parts = build_user_content(row)
    content = [{'type': 'text', 'text': SYSTEM_PROMPT + '\n'}]
    for p in parts:
        if p['is_img']:
            im = p['pil'].copy()
            im.thumbnail((448, 448))
            content.append({'type': 'image', 'image': im})
        if p['is_img']:
            content.append({'type': 'text', 'text': '\n' + p['name'] + ': [Figure image]'})
        else:
            content.append({'type': 'text', 'text': '\n' + p['name'] + ' (caption): ' + p['text']})
    content.append({'type': 'text', 'text': '\nAnswer with one label:'})
    return [{'role': 'user', 'content': content}]
train_ds_list = train_pos.tolist()
val_ds_list = val_pos.tolist()
_log('lists ready: train ' + str(len(train_ds_list)) + ' val ' + str(len(val_ds_list)) + ' (lazy rows from shared frame)')
def _encode(_prompts, _fulls, _imgs, _flat):
    if _flat:
        _imgarg = [im for _sub in _imgs for im in _sub] or None
    else:
        _imgarg = _imgs
    if _imgarg:
        _ef = processor(text=_fulls, images=_imgarg, padding=True, truncation=True, max_length=MAX_SEQ, return_tensors='pt')
        _ep = processor(text=_prompts, images=_imgarg, padding=False, truncation=True, max_length=MAX_SEQ, return_tensors=None)
    else:
        _ef = processor(text=_fulls, padding=True, truncation=True, max_length=MAX_SEQ, return_tensors='pt')
        _ep = processor(text=_prompts, padding=False, truncation=True, max_length=MAX_SEQ, return_tensors=None)
    return _ef, _ep
def collate_fn(features):
    prompts, fulls, img_lists, metas = [], [], [], []
    for _pos in features:
        _row = train.iloc[int(_pos)]
        messages = build_messages(_row)
        images = []
        for turn in messages:
            for part in turn['content']:
                if part.get('type') == 'image':
                    images.append(part['image'])
        p = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        prompts.append(p)
        fulls.append(p + '\n' + str(_row['label']))
        img_lists.append(images)
        metas.append((str(_row['label']), str(_row['pair_type']), str(_row.get('id', ''))))
    try:
        enc_full, enc_prompt = _encode(prompts, fulls, img_lists, len(fulls) == 1)
        _kept = list(range(len(fulls)))
    except Exception as _e0:
        print('Batch encode failed, shrinking: ' + str(_e0)[:160], flush=True)
        enc_full, enc_prompt, _kept = None, None, None
        for _keep in range(len(fulls) - 1, 0, -1):
            _idx = sorted(range(len(fulls)), key=lambda i: len(fulls[i]))[:_keep]
            try:
                enc_full, enc_prompt = _encode([prompts[i] for i in _idx], [fulls[i] for i in _idx], [img_lists[i] for i in _idx], len(_idx) == 1)
                _kept = _idx
                print('WARN: dropped sample(s) from micro-batch', flush=True)
                break
            except Exception as _e1:
                print('Encode failed keep=' + str(_keep) + ' ' + str(_e1)[:160], flush=True)
        if enc_full is None:
            for i in range(len(fulls)):
                try:
                    enc_full, enc_prompt = _encode([prompts[i]], [fulls[i]], [img_lists[i]], True)
                    _kept = [i]
                    print('WARN: flat singleton worked for sample ' + str(i), flush=True)
                    break
                except Exception as _e2:
                    _m = metas[i]
                    print('singleton failed meta=' + str(_m) + ' err=' + str(_e2)[:160], flush=True)
        if enc_full is None:
            _gc = globals().get('_GOOD_CACHE')
            if _gc is not None:
                print('WARN: encoding cached good sample instead', flush=True)
                enc_full, enc_prompt = _encode(_gc[0], _gc[1], _gc[2], True)
            else:
                raise RuntimeError('collate encode failed for all fallbacks')
    if globals().get('_GOOD_CACHE') is None and _kept is not None:
        globals()['_GOOD_CACHE'] = ([prompts[i] for i in _kept], [fulls[i] for i in _kept], [img_lists[i] for i in _kept])
    input_ids = enc_full['input_ids']
    labels = input_ids.clone()
    prompt_lens = [len(ids) for ids in enc_prompt['input_ids']]
    for i, pl in enumerate(prompt_lens):
        pl = min(pl, int(input_ids.shape[1]))
        labels[i, :pl] = -100
    labels[enc_full['attention_mask'] == 0] = -100
    if int((labels != -100).sum()) == 0:
        for i in range(input_ids.shape[0]):
            _L = int(enc_full['attention_mask'][i].sum())
            labels[i, _L-1] = input_ids[i, _L-1]
    enc_full['labels'] = labels
    if 'pixel_values' in enc_full and torch.is_floating_point(enc_full['pixel_values']):
        enc_full['pixel_values'] = enc_full['pixel_values'].to(torch.bfloat16)
    return enc_full
_log('collator ready (answer-only loss)')
model.train()
_dev = torch.device('cuda', LOCAL_RANK)
_batch = collate_fn(train_ds_list[:BATCH])
print({k: (tuple(v.shape) if hasattr(v, 'shape') else type(v).__name__) for k, v in _batch.items()})
print('supervised answer tokens: ' + str(int((_batch['labels'] != -100).sum())))
with torch.no_grad():
    _out = model(**{k: (v.to(_dev) if hasattr(v, 'to') else v) for k, v in _batch.items()})
_log('Forward OK loss: ' + str(round(float(_out.loss), 4)))
del _batch, _out
torch.cuda.empty_cache()
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback, TrainerCallback
class HubPushCallback(TrainerCallback):
    def on_save(self, args, state, control, **kwargs):
        try:
            if RANK != 0 or not HF_TOKEN:
                return
            from huggingface_hub import HfApi
            _api = HfApi(token=HF_TOKEN)
            _ckpt = OUT_DIR + '/checkpoint-' + str(int(state.global_step))
            _api.upload_folder(repo_id=HF_REPO_ID, folder_path=_ckpt, path_in_repo='checkpoints/checkpoint-' + str(int(state.global_step)), commit_message='ckpt ' + str(int(state.global_step)))
            print('Pushed ckpt ' + str(int(state.global_step)), flush=True)
        except Exception as _e:
            print('Push failed: ' + str(_e)[:200], flush=True)
training_args = TrainingArguments(output_dir=OUT_DIR, per_device_train_batch_size=BATCH, per_device_eval_batch_size=1, gradient_accumulation_steps=ACCUM, max_steps=MAX_STEPS, learning_rate=LR, lr_scheduler_type='cosine', warmup_steps=max(10, int(0.05 * MAX_STEPS)), bf16=True, gradient_checkpointing=True, logging_steps=10, save_steps=SAVE_STEPS, save_strategy='steps', eval_strategy='steps', eval_steps=EVAL_STEPS, logging_strategy='steps', save_total_limit=2, load_best_model_at_end=True, metric_for_best_model='eval_loss', greater_is_better=False, push_to_hub=False, report_to='none', remove_unused_columns=False, ddp_find_unused_parameters=True, dataloader_num_workers=0, seed=42)
trainer = Trainer(model=model, args=training_args, train_dataset=train_ds_list, eval_dataset=val_ds_list, data_collator=collate_fn, compute_metrics=None, callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE, early_stopping_threshold=0.001), HubPushCallback()])
_log('eff batch ' + str(BATCH * ACCUM * WORLD) + ' steps/ep ' + str(len(train_ds_list) // (BATCH * ACCUM * WORLD)))
TRAIN_T0 = time.time()
from pathlib import Path as _P
_ckpts = sorted(_P(OUT_DIR).glob('checkpoint-*'), key=lambda q: q.stat().st_mtime)
if not _ckpts and HF_TOKEN:
    try:
        _rf = HfApi().list_repo_files(repo_id=HF_REPO_ID, repo_type='model')
        _steps = sorted([int(p.split('checkpoint-')[-1].split('/')[0]) for p in _rf if p.startswith('checkpoints/checkpoint-') and p.split('checkpoint-')[-1].split('/')[0].isdigit()])
        if _steps:
            _latest = _steps[-1]
            _dl = snapshot_download(repo_id=HF_REPO_ID, allow_patterns=['checkpoints/checkpoint-' + str(_latest) + '/*'], repo_type='model')
            _ckpts = [str(Path(_dl) / ('checkpoints/checkpoint-' + str(_latest)))]
            _log('Resuming from HF checkpoints/checkpoint-' + str(_latest))
        else:
            _log('No HF checkpoints yet - training from scratch')
    except Exception as _e:
        _log('HF resume skipped: ' + str(_e)[:200])
if IS_MAIN:
    open(EVAL_DIR + '/run_config.json', 'w').write(json.dumps({'model': MODEL_ID, 'lora_r': LORA_R, 'lr': LR, 'batch': BATCH, 'accum': ACCUM, 'world': WORLD, 'max_steps': MAX_STEPS, 'repo': HF_REPO_ID}))
    np.save(EVAL_DIR + '/val_ids.npy', DF.iloc[val_pos]['id'].values if 'id' in DF.columns else np.arange(len(val_pos)))
    try:
        _h = HfApi()
        _h.upload_file(path_or_fileobj=EVAL_DIR + '/run_config.json', path_in_repo='run_config.json', repo_id=HF_REPO_ID)
        _h.upload_file(path_or_fileobj=EVAL_DIR + '/val_ids.npy', path_in_repo='val_ids.npy', repo_id=HF_REPO_ID)
    except Exception as _e:
        _log('config push skipped: ' + str(_e)[:200])
trainer.train(resume_from_checkpoint=str(_ckpts[-1]) if _ckpts else None)
_log('best ' + str(trainer.state.best_metric) + ' at ' + str(trainer.state.best_model_checkpoint))
if IS_MAIN:
    trainer.save_model(FINAL_DIR)
    processor.save_pretrained(FINAL_DIR)
    try:
        _api2 = HfApi()
        _api2.upload_folder(repo_id=HF_REPO_ID, folder_path=FINAL_DIR, path_in_repo='adapter_best', commit_message='best eval_loss ' + str(round(float(trainer.state.best_metric), 4)))
        _api2.upload_file(path_or_fileobj=EVAL_DIR + '/run_config.json', path_in_repo='eval_train_config.json', repo_id=HF_REPO_ID)
        print('Pushed adapter_best', flush=True)
    except Exception as _e:
        print('Push best failed ' + str(_e)[:300], flush=True)
_log('TRAIN SCRIPT DONE')


Writing /kaggle/working/train_glm41v.py


In [5]:
# --- 3. Launch DDP: 2 ranks, one full 4-bit model per T4 ---
print('Launching: 2 ranks x GLM-4.1V-9B 4-bit, repo ' + os.environ.get('HF_REPO_ID', '?'))
print('max_steps=' + os.environ.get('MAX_STEPS', '?') + ' eff batch=16 (1x8x2)')
!torchrun --nproc_per_node=2 --master_port=29517 /kaggle/working/train_glm41v.py
print('torchrun finished')


Launching: 2 ranks x GLM-4.1V-9B 4-bit, repo ShivRamSaud/glm41v-9b-lora-astroclimb
max_steps=1536 eff batch=16 (1x8x2)
W0912 06:59:38.783000 96 site-packages/torch/distributed/run.py:793] 
W0912 06:59:38.783000 96 site-packages/torch/distributed/run.py:793] *****************************************
W0912 06:59:38.783000 96 site-packages/torch/distributed/run.py:793] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0912 06:59:38.783000 96 site-packages/torch/distributed/run.py:793] *****************************************
rank 0/2 local_gpu 0 model Rainnighttram/GLM-4.1V-9B-Thinking-bnb-4bit
script rev r11 model-first
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
repo ready ShivRamSaud/glm41v-9b-lora-astroclimb
Note: Environment variable`HF_TOKE

In [6]:
# --- 4. Summary + resume/next ---
print('Model: ' + MODEL_ID + ' LoRA r' + str(LORA_R) + ' DDPx2')
print('Durable state -> HF repo ' + os.environ.get('HF_REPO_ID', '?') + ': checkpoints/* + adapter_best + run files')
print('Local /kaggle/working/glm41v_* is scratch (also kept in session output, but never relied on)')
print('If time runs out: Save Version & Run AGAIN with zero edits - latest HF checkpoint auto-resumes')
print('Next after convergence: generation val eval + submission notebook (same pattern as SmolVLM submit)')


Model: Rainnighttram/GLM-4.1V-9B-Thinking-bnb-4bit LoRA r16 DDPx2
Durable state -> HF repo ShivRamSaud/glm41v-9b-lora-astroclimb: checkpoints/* + adapter_best + run files
Local /kaggle/working/glm41v_* is scratch (also kept in session output, but never relied on)
If time runs out: Save Version & Run AGAIN with zero edits - latest HF checkpoint auto-resumes
Next after convergence: generation val eval + submission notebook (same pattern as SmolVLM submit)
